# A1 · 校準與先驗敏感度 — AUC 高不代表機率可信

> 決策吃的是**機率的絕對值**（門檻、期望損失）。一個 AUC 很高、但校準很差的模型，
> 在決策上仍然危險——因為它說的「90%」可能根本不是 90%。

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import data, bayes_logreg as blr, calibration as cal
DATA = os.path.abspath('../../data/A_medical')

## 1 · 校準分析（步驟 5）

「校準良好」＝在模型說「70%」的那些病人裡，真的約 70% 有病。用 **5-fold 交叉驗證的 out-of-fold 預測**
（297 筆全用上）比較貝葉斯 vs 普通邏輯迴歸，並算 **ECE**（各箱 |實際−預測| 的加權平均）。

In [2]:
Xr, yr, cont_idx, _ = data.load_raw(DATA)
skf = StratifiedKFold(5, shuffle=True, random_state=1)
oof_b, oof_f = np.zeros(len(yr)), np.zeros(len(yr))
for k, (tr, te) in enumerate(skf.split(Xr, yr)):
    sc = StandardScaler().fit(Xr[tr][:, cont_idx])
    Xtr, Xte = Xr[tr].copy(), Xr[te].copy()
    Xtr[:, cont_idx] = sc.transform(Xtr[:, cont_idx]); Xte[:, cont_idx] = sc.transform(Xte[:, cont_idx])
    id_k, _ = blr.fit(Xtr, yr[tr], draws=800, tune=800, chains=2, seed=100+k)
    oof_b[te] = blr.predictive_mean(id_k, Xte)
    oof_f[te] = LogisticRegression(max_iter=2000).fit(Xtr, yr[tr]).predict_proba(Xte)[:,1]
print(f'OOF AUC：貝葉斯={roc_auc_score(yr,oof_b):.3f} · 頻率派={roc_auc_score(yr,oof_f):.3f}')
print(f'ECE  ：貝葉斯={cal.ece(yr,oof_b):.3f} · 頻率派={cal.ece(yr,oof_f):.3f}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [beta, alpha]


Sampling 2 chains for 800 tune and 800 draw iterations (1_600 + 1_600 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


...

Multiprocess sampling (2 chains in 2 jobs)


NUTS: [beta, alpha]


Sampling 2 chains for 800 tune and 800 draw iterations (1_600 + 1_600 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


OOF AUC：貝葉斯=0.906 · 頻率派=0.910
ECE  ：貝葉斯=0.045 · 頻率派=0.047


> **費曼檢驗：校準差的模型，即使 AUC 很高，為什麼在決策上危險？**
> 因為期望損失 $=(1-p)C_{FP}$ 或 $p\,C_{FN}$ 直接吃 $p$ 的絕對值。若模型說「0.9」實際只有「0.6」，
> 你算出的最優門檻與期望損失就全錯——AUC 只看**排序**對不對，不看**數值**準不準。

![校準](../figures/05_calibration.png)

**誠實的結果**：在這份較平衡的資料上，貝葉斯與頻率派校準相近（ECE ≈ 0.045 vs 0.047），兩者都貼近對角線。
貝葉斯的優勢在此更多是**不確定性量化**（NB1 的可信區間），而非校準本身。

## 2 · 先驗敏感度（步驟 1 的敏感度分析）

把先驗從 N(0, 2.5) 換成 N(0, 1)（更緊）與 N(0, 10)（更鬆），看結論穩不穩。

In [3]:
ds = data.load_heart(DATA, test_size=0.25, seed=0)
for psd in [1.0, 2.5, 10.0]:
    idp, _ = blr.fit(ds.X_train, ds.y_train, prior_sd=psd, draws=1000, tune=1000, chains=2, seed=7)
    auc = roc_auc_score(ds.y_test, blr.predictive_mean(idp, ds.X_test))
    print(f'prior_sd={psd:>4}: 測試 AUC={auc:.3f}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [beta, alpha]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


...

Multiprocess sampling (2 chains in 2 jobs)


NUTS: [beta, alpha]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 2 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


prior_sd=10.0: 測試 AUC=0.906


![先驗敏感度](../figures/06_prior_sensitivity.png)

**結論穩健**：較緊的先驗會把係數往 0 收縮（正則化，圖上可見），但**符號、特徵排序、預測 AUC 幾乎不變**（0.906–0.911）。
換句話說，決策不依賴先驗的精確選擇——這正是敏感度分析要證明的。

## 3 · 限制與誠實的部分

- **校準**：此資料較平衡，貝葉斯與頻率派校準相近；貝葉斯的主要價值在不確定性量化，別過度宣稱它「校準更好」。
- **樣本量**：僅 297 筆，測試集 75 筆——決策門檻的實際損失估計有抽樣噪音（已用 5-fold 緩解校準估計）。
- **損失矩陣是假設**：$C_{FN}/C_{FP}$ 與 $C_{reject}$ 是外部輸入，結論隨它們改變——這正是決策理論的重點：把假設攤在陽光下。
- **特徵編碼**：名目類別已 one-hot，但未做交互作用/非線性；重點在決策層，不在特徵工程。

## 重點

> AUC 衡量**排序**，校準衡量**數值**。做決策時吃的是數值，所以校準與不確定性量化才是關鍵。
> 貝葉斯把「我對這個機率有多確定」也一起給你——這正是從「預測」跨到「決策」缺的那一塊。